# 📝 Neo4j 시작하기 과제 LV1(기초): 결과 읽기·모델 판별

> 이 단원의 핵심을 **하나씩** 확인합니다. 제공된 조회 결과를 파이썬으로 읽어 답을 찾고, 그래프 모델(노드·관계·속성)을 판별합니다.

## 풀이 방법
1. 맨 위 **준비 셀 → 점검 셀**을 먼저 실행하세요(Movies 그래프가 적재돼 있어야 합니다).
2. 각 문제는 위쪽 셀이 데이터를 먼저 가져옵니다(실행만 하면 됩니다). 여러분은 그 결과를 **파이썬으로 후처리**해 답안 셀을 채웁니다.
3. **자가채점 셀**로 확인하세요(✅ 통과!). 6·11번은 서술형이라 자가채점이 없습니다.

- **Cypher 는 거의 쓰지 않습니다.** 딱 한 문제(3번)에서 제공된 쿼리의 **숫자 하나만** 바꿔 다시 물어봅니다. 나머지는 결과 해석과 모델 이해입니다.
- 자가채점은 그래프에 **다시 물어본 값**과 비교합니다. 제공 셀의 결과를 **파이썬으로 가공해** 답을 만드세요(눈으로 읽어 옮겨 적으면 채점은 지나가도 이 단원의 훈련이 되지 않습니다).

화이팅!

아래 준비 셀과 점검 셀을 먼저 실행하세요(내용은 이해하지 않아도 됩니다. 실행만 하면 됩니다).

In [ ]:
# Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 이 단원은 그래프를 조회만 합니다(그래프를 바꾸지 않습니다).
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase   # 파이썬용 공식 드라이버. 이 클래스로 접속 통로를 연다

# 1) 접속 정보 읽기: .env 에 적힌 값을 환경변수로 올린다(파일이 없으면 조용히 넘어간다)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 두 번째 인자는 .env 에 그 키가 없을 때 쓰는 기본값이다(로컬 Desktop 의 표준 주소·사용자).
# .env 를 못 읽어도 에러가 아니라 이 값으로 조용히 넘어가니, 이 셀 마지막 줄에 찍히는
# 주소가 실습 전용 DB 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
# 2) 드라이버 만들기: 접속 통로 하나를 노트북 전체가 나눠 쓴다(쿼리마다 새로 만들지 않는다)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 실제로 붙어 본다. 인스턴스가 꺼져 있거나 비밀번호가 틀리면 여기서 에러가 난다


# 3) 수업 내내 쓰는 헬퍼: 쿼리를 보내고 결과를 파이썬 자료형으로 바꿔 준다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    with driver.session() as session:
        # 세션은 with 블록을 벗어나면 자동으로 닫힌다. record.data() 가 결과 한 행을 dict 로 바꾼다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)   # 이 줄이 찍히면 연결까지 성공한 것이다

In [ ]:
# Movies 그래프가 적재돼 있는지 점검: 실행만 하세요(그래프를 바꾸지 않습니다).
# MATCH (n) 은 레이블을 가리지 않고 모든 노드를 고른다. count(n) 결과는 한 행이라 [0] 으로 dict 를 꺼낸다
_n = run_cypher("MATCH (n) RETURN count(n) AS cnt")[0]["cnt"]   # 이름 앞 밑줄은 이 셀에서만 쓰는 임시 변수라는 표시
print("연결된 그래프의 노드 수:", _n)   # 171 이면 준비 완료, 0 이면 아직 적재 전이다

## 1. 영화가 몇 편일까
**배경**: 그래프에 영화(`Movie`)가 몇 편 들어 있는지 셉니다. 제공 셀이 영화 제목을 리스트로 가져옵니다.

**요구사항**:
- 제공된 `movie_rows`(영화 제목들)의 **개수**를 변수 **`n_movies`** 에 담으세요.

**예시**: `n_movies` 는 `12` 같은 **정수 하나**입니다(`12` 는 형태를 보이는 값일 뿐이니 정확한 값은 직접 세어 보세요).

<details><summary>힌트</summary>

```text
접근방법:
- 리스트의 길이를 len 으로 센다.

세부구현:
1. 리스트의 길이를 세는 내장 함수로 개수를 구해 n_movies 에 담는다.
```

</details>

In [ ]:
# 영화 제목을 가져옵니다(실행만 하세요).
movie_rows = run_cypher("MATCH (m:Movie) RETURN m.title AS title ORDER BY m.title")
print('가져온 행 수:', len(movie_rows))

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 그래프에 다시 물어본 값과 비교합니다.
_expected = run_cypher("MATCH (m:Movie) RETURN count(m) AS c")[0]['c']
assert n_movies == _expected, '영화 편수가 그래프에 물어본 값과 다릅니다'
print('✅ 통과!')

## 2. 1964년에 태어난 인물 찾기
**배경**: 인물마다 출생연도(`born`) 속성이 있습니다(없는 사람도 있습니다). 특정 연도에 태어난 사람을 골라냅니다.

**요구사항**:
- 제공된 `people`(이름·출생연도)에서 `born` 이 **1964** 인 사람의 **이름**만 골라 리스트 **`born_1964`** 에 담으세요. **이름 오름차순**(제공 결과가 이미 그 순서입니다).

**예시**: `born_1964` 는 `['이름', ...]` 형태의 리스트이고, 해당하는 사람은 **1명**입니다(누구인지는 직접 찾으세요).

<details><summary>힌트</summary>

```text
접근방법:
- born 이 1964 인 row 만 남기고 그 이름을 모은다.

세부구현:
1. 리스트 컴프리헨션에서 출생연도 키의 값이 1964 인 행만 남긴다.
2. 남은 행에서 이름 키의 값만 모아 born_1964 를 만든다.
```

</details>

In [ ]:
# 인물 이름과 출생연도를 가져옵니다(실행만 하세요).
people = run_cypher("MATCH (p:Person) WHERE p.born IS NOT NULL "
                    "RETURN p.name AS name, p.born AS born ORDER BY p.name")
print('출생연도가 있는 인물 수:', len(people))

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 그래프에 다시 물어본 목록과 비교합니다.
_expected = [r['name'] for r in run_cypher("MATCH (p:Person) WHERE p.born = 1964 "
                                          "RETURN p.name AS name ORDER BY p.name")]
assert len(born_1964) == 1, '인원수가 맞지 않습니다'
assert born_1964 == _expected, '1964년생 이름 목록(이름 오름차순)이 다릅니다'
print('✅ 통과!')

## 3. 쿼리에서 연도만 바꿔 다시 묻기
**배경**: Cypher 를 쓰는 법은 다음 단원에서 배우지만, **이미 있는 쿼리에서 값 하나를 바꾸는 것**은 지금도 할 수 있습니다. 교안에서 영화 제목을 바꿔 봤던 것과 같은 방식입니다.

**요구사항**:
- 아래 제공 셀의 쿼리를 **그대로 복사**한 뒤 **연도 숫자만** `1964` → `1967` 로 바꿔 실행하세요(다른 부분은 손대지 않습니다).
- 그 결과에서 **이름만** 모은 리스트를 변수 **`born_1967`** 에 담으세요. 순서는 **이름 오름차순**(제공 쿼리의 `ORDER BY` 를 그대로 두면 됩니다).

**예시**: `born_1967` 은 `['이름', ...]` 형태이고, 해당하는 사람은 **7명**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 제공 쿼리를 복사해 연도만 바꾸고, 결과 dict 에서 이름 키만 꺼내 모은다.

세부구현:
1. 쿼리 문자열에서 연도 숫자만 1967 로 바꿔 다시 실행한다.
2. 돌아온 각 행에서 이름 키의 값만 모아 born_1967 에 담는다.
```

</details>

In [ ]:
# 이 쿼리를 복사해 연도만 바꿔 쓰세요. 이 셀 자체는 실행만 하세요.
rows_1964 = run_cypher("MATCH (p:Person) WHERE p.born = 1964 "
                       "RETURN p.name AS name ORDER BY p.name")
print('1964년생:', len(rows_1964), '명')

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 그래프에 다시 물어본 목록과 비교합니다.
_expected = [r['name'] for r in run_cypher("MATCH (p:Person) WHERE p.born = 1967 "
                                          "RETURN p.name AS name ORDER BY p.name")]
assert len(born_1967) == 7, '인원수가 맞지 않습니다'
assert born_1967 == _expected, '1967년생 이름 목록(이름 오름차순)이 다릅니다'
print('✅ 통과!')

## 4. 특정 영화의 출연진 명단
**배경**: 한 영화에 누가 출연했는지 명단을 만듭니다. 제공 셀이 영화 **Top Gun** 의 출연 정보를 가져옵니다.

**요구사항**:
- 제공된 `topgun_rows` 에서 **배우 이름만** 모은 리스트를 변수 **`cast_names`** 에 담으세요(순서는 제공 결과 그대로 = 이름 오름차순).
- 이어서 그 인원수를 변수 **`n_cast`** 에 담으세요.

**예시**: `cast_names` 는 `['이름', ...]` 형태이고 출연진은 **6명**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 각 row 에서 이름 키를 꺼내 리스트로 모으고, 그 길이를 센다.

세부구현:
1. 각 행에서 이름 키의 값만 모아 cast_names 를 만든다.
2. 그 리스트의 길이를 세어 n_cast 에 담는다.
```

</details>

In [ ]:
# Top Gun 에 출연한 배우를 가져옵니다(실행만 하세요).
topgun_rows = run_cypher("MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title:'Top Gun'}) "
                         "RETURN p.name AS name ORDER BY p.name")
print('가져온 행 수:', len(topgun_rows))

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 그래프에 다시 물어본 명단과 비교합니다.
_expected = [r['name'] for r in run_cypher("MATCH (p:Person)-[:ACTED_IN]->"
                                          "(:Movie {title:'Top Gun'}) "
                                          "RETURN p.name AS name ORDER BY p.name")]
assert cast_names == _expected, '출연진 명단이 그래프에 물어본 명단과 다릅니다'
assert n_cast == 6, '출연진 수가 맞지 않습니다'
print('✅ 통과!')

## 5. 한 배우의 1990년대 출연작
**배경**: 배우 **Cuba Gooding Jr.** 의 출연작 중 **1990년대**(1990~1999) 작품만 골라냅니다. 아래 셀이 제목과 개봉연도를 함께 가져옵니다. **2000년 작품이 섞여 있으니 경계를 조심하세요.**

**요구사항**:
- `actor_rows` 에서 `released` 가 **1990 이상 2000 미만**인 작품의 **제목**만 모아 리스트 **`films_90s`** 에 담으세요(제목 오름차순 = 앞 셀 결과 순서).

**예시**: `films_90s` 는 `['제목', ...]` 형태이고 해당 작품은 **3편**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 개봉연도 범위 조건으로 거른 뒤 제목만 모은다.

세부구현:
1. 리스트 컴프리헨션에서 개봉연도가 1990 이상이고 2000 미만인 행만 남긴다(2000년은 1990년대가 아니다).
2. 남은 행에서 제목 키의 값만 모아 films_90s 에 담는다.
```

</details>

In [ ]:
# Cuba Gooding Jr. 이(가) 출연한 영화의 제목·개봉연도를 가져옵니다(실행만 하세요).
actor_rows = run_cypher("MATCH (:Person {name:'Cuba Gooding Jr.'})-[:ACTED_IN]->(m:Movie) "
                        "RETURN m.title AS title, m.released AS released ORDER BY m.title")
for row in actor_rows:
    print(row)   # 2000년작이 섞여 있으니 경계를 조심한다

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 그래프에 다시 물어본 목록과 비교합니다.
_expected = [r['title'] for r in run_cypher(
    "MATCH (:Person {name:'Cuba Gooding Jr.'})-[:ACTED_IN]->(m:Movie) "
    "WHERE m.released >= 1990 AND m.released < 2000 RETURN m.title AS title ORDER BY m.title")]
assert len(films_90s) == 3, '1990년대 작품 편수가 맞지 않습니다(2000년작을 넣지 않았는지 확인하세요)'
assert films_90s == _expected, '1990년대 작품 목록이 다릅니다'
print('✅ 통과!')

## 6. (서술형) 이 값은 어디에 붙어 있나
**배경**: 그래프 모델을 이해했는지 확인합니다. 이 문제는 **자가채점이 없습니다**. 아래 markdown 셀에 직접 서술하고, 정답 노트북의 모범 서술과 비교하세요.

**요구사항**: 다음 세 가지가 각각 **관계 / 노드 속성 / 관계 속성** 중 무엇인지, 그리고 **왜** 그런지 한 줄씩 쓰세요. 이어서 각 항목이 **표(RDB)였다면 어디에 담겼을지**도 한 줄로 덧붙이세요.

1. 영화의 `tagline`(홍보 문구)
2. `REVIEWED`(평가함)
3. 리뷰의 `rating`(점수)

**힌트**: 개체를 잇는 연결(동사)인지, 개체 하나에 딸린 값인지, **그 연결에만 딸린 값**인지로 구분합니다. 값이 주체마다 달라지면 노드가 아니라 연결에 붙습니다.

**자가 점검**: 답을 쓴 뒤 스스로 확인하세요.
- [ ] 세 항목 모두 **관계 / 노드 속성 / 관계 속성 중 하나**를 분명히 골랐다.
- [ ] `rating` 을 노드 속성과 가른 근거가 "주체마다 값이 달라진다" 로 설명된다.
- [ ] `REVIEWED` 는 **방향**(누가 → 무엇)까지 밝혔다.
- [ ] 세 항목 모두 **표(RDB)의 어느 자리**에 대응하는지 적었다.

*(여기에 자신의 답을 서술하세요)*

1. `tagline` → 
2. `REVIEWED` → 
3. `rating` → 

## 7. 관계 속성 `roles` 다루기
**배경**: 배우가 한 영화에서 맡은 배역(`roles`)은 **출연 관계(`ACTED_IN`)의 속성**이고, **리스트**입니다(한 영화에서 여러 배역을 맡을 수 있어서). 제공 셀이 Meg Ryan 이 **Joe Versus the Volcano** 에서 맡은 배역 리스트를 가져옵니다.

**요구사항**:
- 제공된 `mr_roles`(배역 리스트)에서 **배역 개수**(리스트 길이)를 변수 **`n_roles`** 에 담으세요.

**예시**: `n_roles` 는 정수 하나입니다(제공 셀이 출력한 리스트를 보고 직접 세어 보세요).

<details><summary>힌트</summary>

```text
접근방법:
- roles 는 리스트다. 그 길이를 센다.

세부구현:
1. 리스트의 길이를 세는 내장 함수로 배역 수를 구해 n_roles 에 담는다.
```

</details>

In [ ]:
# Meg Ryan 이 Joe Versus the Volcano 에서 맡은 배역(관계 속성)을 가져옵니다(실행만 하세요).
mr_roles = run_cypher("MATCH (:Person {name:'Meg Ryan'})-[r:ACTED_IN]->"
                      "(:Movie {title:'Joe Versus the Volcano'}) RETURN r.roles AS roles")[0]['roles']
print('배역 리스트:', mr_roles)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 그래프에 다시 물어본 배역 리스트의 길이와 비교합니다.
_expected = run_cypher("MATCH (:Person {name:'Meg Ryan'})-[r:ACTED_IN]->"
                       "(:Movie {title:'Joe Versus the Volcano'}) RETURN r.roles AS roles")[0]['roles']
assert isinstance(n_roles, int), 'n_roles 에는 배역 개수(정수 하나)를 담으세요'
assert n_roles == len(_expected), \
    '배역 개수가 다릅니다(제공된 배역 리스트의 길이를 세었는지 확인하세요)'
print('✅ 통과!')

## 8. 출생연도가 없는 인물 세기
**배경**: 속성은 **늘 있는 게 아닙니다**. 어떤 인물 노드에는 `born`(출생연도)이 없습니다. 제공 셀이 **모든** 인물의 이름과 출생연도(없으면 `None`)를 가져옵니다.

**요구사항**:
- 제공된 `all_people` 에서 `born` 이 **`None`**(값이 없는) 인 인물의 **수**를 변수 **`n_no_born`** 에 담으세요.

**예시**: `n_no_born` 은 정수 하나입니다(전체 인물 수보다 훨씬 작습니다).

<details><summary>힌트</summary>

```text
접근방법:
- born 이 None 인 row 만 세면 된다.

세부구현:
1. 출생연도 키의 값이 없는(None 인) 행이 몇 개인지 센다(sum 이나 컴프리헨션 + 길이).
2. 그 수를 n_no_born 에 담는다.
```

</details>

In [ ]:
# 모든 인물의 이름과 출생연도(없으면 None)를 가져옵니다(실행만 하세요).
all_people = run_cypher("MATCH (p:Person) RETURN p.name AS name, p.born AS born ORDER BY p.name")
print('전체 인물 수:', len(all_people))

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 그래프에 다시 물어본 값과 비교합니다.
_expected = run_cypher("MATCH (p:Person) WHERE p.born IS NULL RETURN count(p) AS c")[0]['c']
assert n_no_born == _expected, '출생연도가 없는 인물 수가 다릅니다'
print('✅ 통과!')

## 9. 한 쌍을 잇는 관계가 3종인 쌍 찾기
**배경**: 교안에서 본 것처럼, 같은 (사람, 영화) 쌍을 **여러 관계**가 잇기도 합니다(각본을 쓰고 제작하고 감독까지). 제공 셀이 관계가 **2종 이상**인 쌍을 모두 가져옵니다.

**요구사항**:
- 제공된 `multi_rows`(각 원소에 `person`·`movie`·`types` 포함)에서 `types` 의 길이가 **3 이상**인 쌍만 골라, `(person, movie)` **튜플**의 리스트를 변수 **`triples`** 에 담으세요. 순서는 **사람 이름 오름차순**(제공 결과가 이미 그 순서입니다).
- 이어서 `triples` 의 **개수**를 변수 **`n_triple`** 에 담으세요.

**예시**: `triples` 는 `[('사람 이름', '영화 제목'), ...]` 형태의 리스트입니다(누가 어느 작품인지는 직접 찾으세요).

<details><summary>힌트</summary>

```text
접근방법:
- types 리스트 길이가 3 이상인 row 만 남기고, 그 row 의 사람과 영화를 튜플로 묶는다.

세부구현:
1. 관계 이름 리스트의 길이가 3 이상인 행만 남긴다.
2. 남은 행의 사람 키와 영화 키를 튜플 하나로 묶어 triples 에 모은다.
3. 그 목록의 길이를 n_triple 에 담는다.
```

</details>

In [ ]:
# 한 쌍을 잇는 관계가 2종 이상인 (사람, 영화) 쌍(실행만 하세요).
multi_rows = run_cypher("MATCH (p:Person)-[r]->(m:Movie) WITH p, m, collect(DISTINCT type(r)) AS types "
                        "WHERE size(types) >= 2 RETURN p.name AS person, m.title AS movie, "
                        "types AS types ORDER BY p.name")
print('관계가 2종 이상인 쌍:', len(multi_rows))

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 그래프에 다시 물어본 목록과 비교합니다.
_expected = [(r['person'], r['movie']) for r in run_cypher(
    "MATCH (p:Person)-[r]->(m:Movie) WITH p, m, collect(DISTINCT type(r)) AS types "
    "WHERE size(types) >= 3 RETURN p.name AS person, m.title AS movie ORDER BY p.name")]
assert all(isinstance(t, tuple) for t in triples), \
    'triples 의 원소는 (사람, 영화) 튜플이어야 합니다'
assert triples == _expected, \
    '관계가 3종인 (사람, 영화) 쌍 목록이 다릅니다(사람 이름 오름차순으로 담았는지 확인하세요)'
assert n_triple == len(_expected), 'n_triple 은 triples 의 개수여야 합니다'
print('✅ 통과!')

## 10. 팔로우당하는 사람 모으기
**배경**: 관계가 잇는 두 끝이 늘 다른 종류인 것은 아닙니다. Movies 그래프의 `FOLLOWS` 는 **사람에서 사람으로** 향합니다. 양쪽 끝이 모두 사람이라, **어느 쪽 끝을 꺼내는가**가 곧 **누구를 묻는가**가 됩니다. 제공 셀이 팔로우 관계를 모두 가져옵니다.

**요구사항**:
- 제공된 `follows_rows`(각 원소에 `follower`·`followee` 포함)에서 **팔로우를 당하는 쪽**(`followee`)만 모아 **집합(set)** `followees` 에 담으세요(중복 없이).
- 이어서 그 인원수를 변수 **`n_followees`** 에 담으세요.

**예시**: `followees` 는 `{'이름', ...}` 형태의 집합이고, 인원수는 팔로우 관계 수보다 **적습니다**(한 사람이 여러 사람에게 팔로우당할 수 있어서).

<details><summary>힌트</summary>

```text
접근방법:
- 각 row 에서 팔로우당하는 쪽 키만 꺼내 집합으로 모은다. 집합이라 중복은 저절로 사라진다.

세부구현:
1. 각 행에서 팔로우당하는 쪽 키의 값을 꺼낸다.
2. 중괄호 집합 컴프리헨션으로 모아 followees 를 만든다.
3. 그 집합의 크기를 n_followees 에 담는다.
```

</details>

In [ ]:
# 사람에서 사람으로 향하는 FOLLOWS 관계를 가져옵니다(실행만 하세요).
follows_rows = run_cypher("MATCH (a:Person)-[:FOLLOWS]->(b:Person) "
                          "RETURN a.name AS follower, b.name AS followee ORDER BY follower")
for row in follows_rows:
    print(row)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 그래프에 다시 물어본 집합과 비교합니다.
_expected = {r['followee'] for r in run_cypher("MATCH (:Person)-[:FOLLOWS]->(b:Person) "
                                              "RETURN b.name AS followee")}
assert isinstance(followees, set), \
    'followees 는 집합(set)이어야 합니다. 중괄호로 모으면 중복이 사라집니다'
assert followees == _expected, \
    '팔로우당하는 사람 집합이 다릅니다(follower 와 followee 를 바꿔 꺼내지 않았는지 확인하세요)'
assert n_followees == len(_expected), 'n_followees 는 followees 의 인원수여야 합니다'
print('✅ 통과!')

## 11. (서술형) `REVIEWED` 의 방향
**배경**: 앞 문제에서 방향이 뜻을 가른다는 것을 보았습니다. 이 문제는 **자가채점이 없습니다**. 아래 markdown 셀에 직접 서술하고, 정답 노트북의 모범 서술과 비교하세요.

**요구사항**: 평가 관계 `REVIEWED` 에 대해 다음 두 가지를 서술하세요.

1. `REVIEWED` 는 어느 쪽에서 어느 쪽으로 향하나요? 그렇게 판단한 근거는 무엇인가요(교안에서 실행한 결과나 상식 중 무엇을 근거로 삼아도 좋습니다)?
2. 만약 이 관계를 **반대 방향으로** 저장했다면, 어떤 조회가 이상해질지 구체적인 예를 들어 설명하세요.

**자가 점검**: 답을 쓴 뒤 스스로 확인하세요.
- [ ] 방향을 **누가 → 무엇** 형태로 분명히 적었다.
- [ ] 근거가 "평가하는 쪽과 평가받는 쪽" 중 어느 것이 주체인지로 설명된다.
- [ ] 반대로 저장했을 때 **이상해지는 조회를 한 문장으로** 예시했다.

*(여기에 자신의 답을 서술하세요)*

1. `REVIEWED` 의 방향과 근거: 
2. 반대 방향으로 저장했다면: 

---
수고했어요! LV1 에서 제공된 조회 결과를 **파이썬으로 읽어** 답을 찾고, 쿼리의 값 하나를 바꿔 다시 물어보고, 노드·관계·속성 모델을 **판별**했습니다. LV2 에서는 이 결과들을 **pandas·collections 로 가공**해 분포와 순위를 뽑습니다.